# Parse the MODS Records into a Hearing Table

| Item | Description |
|---|---|
| Goal | Turn 36,378 MODS records into one row per hearing with its committees, members, witnesses, sitting dates, and whether the transcript was digitized. Later stages select on these fields, so this is where the corpus becomes a sample with known properties. |
| Input | `Raw/manifest/packages_all.parquet` from D1.1.1. From D1.1.2: `Raw/status/fetch_summary.parquet`, `Raw/<year>/mods/<packageId>.xml`, and `Raw/<year>/html/<packageId>.htm` (read only for its size and the availability marker). |
| Output | In the D1.1 Interim folder: `hearings.parquet`, one row per package; `hearing_committees.parquet`, one row per hearing and committee; `hearing_members.parquet`, one row per hearing and member; `coverage_by_year.csv`, usable hearings per year. |
| Variables | • `n_committees`, `n_members`, `n_members_with_bioguide`, `n_witnesses`: counts taken from the MODS record<br>• `committee_id`, `committee_name`: the first committee listed; every committee stays in the list column `committees`<br>• `has_transcript`: true when the HTML file holds a transcript and not only a cover page |
| Run | Last run 2026-08-20; the cell outputs shown are from that run. Not rerun: D1.1.4 and every later stage were built on these tables. The files now sit in the Interim folder named above. |

## Step 1. Import packages

`lxml` reads the XML records.

In [ ]:
from pathlib import Path

import pandas as pd
from lxml import etree

## Step 2. Define file locations

In [ ]:
PROJECT_FOLDER = Path.cwd().resolve().parents[1]  # <section>/<subfolder>/ sits two levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("D1 - Data Collection and Processing/D1.1 - GovInfo Congressional Hearings")
NOTEBOOK_NAME = "D1.1.3 - GovInfo - Parse MODS Metadata"
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
MANIFEST_FOLDER = RAW_FOLDER / "manifest"  # written by D1.1.1
STATUS_FOLDER = RAW_FOLDER / "status"  # written by D1.1.2

INTERIM_FOLDER.mkdir(parents=True, exist_ok=True)

## Step 3. Load the manifest and the download status

The work list comes from the status log, not from a folder listing. A missing file is then a recorded fact, not an invisible gap.

In [1]:
manifest = pd.read_parquet(MANIFEST_FOLDER / "packages_all.parquet")
status = pd.read_parquet(STATUS_FOLDER / "fetch_summary.parquet")

print(f"manifest: {len(manifest):,} packages")
print(f"status  : {len(status):,} fetch outcomes")
print(status["status"].value_counts().to_string())

manifest: 36,378 packages
status  : 73,221 fetch outcomes
status
ok         72346
missing      875


## Step 4. Define how one MODS record and one HTML file are read

A MODS record holds the title, the issue date, and three sibling `extension` blocks. The blocks carry provenance, package identity with the sitting dates, and content with committees, members, and witnesses. Every field is found by searching the whole tree, because which block holds it is a fact about GovInfo. All committees are kept, not only the first, because joint hearings list more than one.

In [ ]:
NS = {"m": "http://www.loc.gov/mods/v3"}
NO_TEXT_MARKER = b"<TEXT NOT AVAILABLE>"
STUB_MAX_BYTES = 20_000  # a cover page is a few KB; anything larger is a transcript


def element_text(element):
    """Stripped text of an element, or None."""
    if element is None or element.text is None:
        return None
    return element.text.strip() or None


def find_text(root, path):
    return element_text(root.find(path, NS))


def named(element, name_type):
    """The <name type="..."> child of a congCommittee or congMember."""
    return element_text(element.find(f'm:name[@type="{name_type}"]', NS))


def parse_mods(path):
    """One MODS file to one row: identity, dates, committees, members, witnesses, and their counts."""
    path = Path(path)
    root = etree.parse(str(path)).getroot()

    committees = []  # every committee, in document order
    for el in root.iterfind(".//m:congCommittee", NS):
        committees.append({
            "authorityId": el.get("authorityId"), "chamber": el.get("chamber"), "congress": el.get("congress"),
            "type": el.get("type"),
            "name_standard": named(el, "authority-standard"), "name_short": named(el, "authority-short"),
        })
    members = []  # bioGuideId is sometimes absent; the member is kept with a null id so the gap can be counted
    for el in root.iterfind(".//m:congMember", NS):
        members.append({
            "bioGuideId": el.get("bioGuideId"), "chamber": el.get("chamber"), "congress": el.get("congress"),
            "party": el.get("party"), "state": el.get("state"), "role": el.get("role"),
            "name_parsed": named(el, "parsed"), "name_lnf": named(el, "authority-lnf"),
            "name_fnf": named(el, "authority-fnf"), "name_other": named(el, "authority-other"),
        })
    # A witness line is free text ("Surname, Given name, Title, Organization") and is kept as written.
    witnesses = [t for t in (element_text(el) for el in root.iterfind(".//m:witness", NS)) if t]
    held = [t for t in (element_text(el) for el in root.iterfind(".//m:heldDate", NS)) if t]

    primary = committees[0] if committees else {}
    return {
        "packageId": find_text(root, ".//m:recordInfo/m:recordIdentifier") or find_text(root, ".//m:accessId") or path.stem,
        "title": find_text(root, "m:titleInfo/m:title"),
        "dateIssued": find_text(root, "m:originInfo/m:dateIssued"),
        "docClass": find_text(root, ".//m:docClass"),
        "chamber": find_text(root, ".//m:chamber"),
        "congress": find_text(root, ".//m:congress"),
        "session": find_text(root, ".//m:session"),
        "number": find_text(root, ".//m:number"),
        "granuleClass": find_text(root, ".//m:granuleClass"),
        "preferredCitation": find_text(root, ".//m:preferredCitation"),
        "sudoc": find_text(root, 'm:classification[@authority="sudocs"]'),
        "branch": find_text(root, ".//m:branch"),
        "dateIngested": find_text(root, ".//m:dateIngested"),
        "isNomination": find_text(root, ".//m:isNomination"),
        "isErrata": find_text(root, ".//m:isErrata"),
        "heldDates": held,
        "heldDate_first": held[0] if held else None,
        "committees": committees,
        "members": members,
        "witnesses": witnesses,
        "committee_name": primary.get("name_standard"),
        "committee_id": primary.get("authorityId"),
        "n_committees": len(committees),
        "n_members": len(members),
        "n_members_with_bioguide": sum(1 for m in members if m["bioGuideId"]),
        "n_witnesses": len(witnesses),
        "n_held_dates": len(held),
    }


def transcript_status(path):
    """Whether the HTML file exists, its size, and whether it holds a transcript rather than a cover page."""
    path = Path(path)
    if not path.exists():
        return {"exists": False, "bytes": 0, "has_text": False}
    size = path.stat().st_size
    if size > STUB_MAX_BYTES:
        return {"exists": True, "bytes": size, "has_text": True}
    return {"exists": True, "bytes": size, "has_text": NO_TEXT_MARKER not in path.read_bytes()}

## Step 5. Parse every MODS record

A malformed record is collected, not raised, so one bad file cannot stop a run over 36,000 files. The number of failures is printed here and checked in Step 11.

In [2]:
mods_ok = status.loc[(status["kind"] == "mods") & status["status"].isin(["ok", "skipped"])]
missing_mods = set(manifest["packageId"]) - set(mods_ok["packageId"])
print(f"{len(mods_ok):,} MODS records to parse, {len(missing_mods):,} packages without one")

records = []
failures = []
for path in mods_ok["path"]:
    try:
        records.append(parse_mods(path))
    except Exception as exc:  # a malformed record is data, not a reason to stop
        failures.append({"path": path, "error": f"{type(exc).__name__}: {exc}"})

hearings = pd.DataFrame.from_records(records)
print(f"parsed {len(hearings):,} records, {len(failures)} failed")
if failures:
    print(pd.DataFrame(failures).head(10).to_string(index=False))

36,378 MODS records to parse, 0 packages without one


parsed 36,378 records, 0 failed


## Step 6. Attach the transcript's availability

Three columns per package: whether the HTML file is on disk, its size, and whether it holds a transcript. GovInfo writes `<TEXT NOT AVAILABLE>` where the body of an undigitized hearing would be. Files above 20 KB are transcripts, and only the small ones are scanned for the marker. A file listing cannot answer the third question.

In [3]:
year_of = dict(zip(manifest["packageId"], manifest["query_year"]))
hearings["year"] = hearings["packageId"].map(year_of)

html_paths = [RAW_FOLDER / str(year) / "html" / f"{package_id}.htm"
              for package_id, year in zip(hearings["packageId"], hearings["year"])]
text_status = pd.DataFrame.from_records([transcript_status(p) for p in html_paths])
hearings["html_bytes"] = text_status["bytes"].to_numpy()
hearings["has_transcript"] = text_status["has_text"].to_numpy()
hearings["html_on_disk"] = text_status["exists"].to_numpy()

print(f"{hearings['html_on_disk'].sum():,} transcripts on disk")
print(f"{hearings['has_transcript'].sum():,} contain text ({1 - hearings['has_transcript'].mean():.1%} are cover-page stubs)")

35,503 transcripts on disk
35,125 contain text (3.4% are cover-page stubs)


## Step 7. Build the committee and member tables

The two tables are exploded from the list columns on the hearing row, so they cannot disagree with it.

In [4]:
committee_rows = [{"packageId": row.packageId, "position": i, **committee}
                  for row in hearings.itertuples() for i, committee in enumerate(row.committees)]
member_rows = [{"packageId": row.packageId, **member}
               for row in hearings.itertuples() for member in row.members]

hearing_committees = pd.DataFrame.from_records(committee_rows)
hearing_members = pd.DataFrame.from_records(member_rows)

print(f"hearing-committee pairs: {len(hearing_committees):,} across {hearing_committees['packageId'].nunique():,} hearings")
print(f"hearing-member pairs   : {len(hearing_members):,} across {hearing_members['packageId'].nunique():,} hearings")
print(f"distinct committees    : {hearing_committees['name_standard'].nunique():,}")
print(f"distinct bioGuideIds   : {hearing_members['bioGuideId'].nunique():,}")

hearing-committee pairs: 37,258 across 36,366 hearings
hearing-member pairs   : 1,031,089 across 31,930 hearings
distinct committees    : 83
distinct bioGuideIds   : 1,554


## Step 8. Count usable hearings by year

A hearing is usable when it has a committee to select on, members to attribute statements to, and a transcript to read. This table is the denominator for every count a later stage reports. A series that rises because coverage rises is not a finding.

In [5]:
flags = pd.DataFrame({
    "year": hearings["year"],
    "has_committee": hearings["n_committees"] > 0,
    "has_members": hearings["n_members"] > 0,
    "has_bioguide": hearings["n_members_with_bioguide"] > 0,
    "has_transcript": hearings["has_transcript"],
})
flags["usable"] = flags["has_committee"] & flags["has_members"] & flags["has_transcript"]

coverage = flags.groupby("year").agg(
    hearings=("has_committee", "size"),
    with_committee=("has_committee", "sum"),
    with_members=("has_members", "sum"),
    with_bioguide=("has_bioguide", "sum"),
    with_transcript=("has_transcript", "sum"),
    usable=("usable", "sum"),
)
for column in ("with_committee", "with_members", "with_transcript", "usable"):
    coverage[column.replace("with_", "") + "_pct"] = (coverage[column] / coverage["hearings"]).map("{:.0%}".format)

print(coverage[["hearings", "with_committee", "with_members", "with_transcript", "usable", "usable_pct"]].to_string())

      hearings  with_committee  with_members  with_transcript  usable usable_pct
year                                                                            
1995       253             253             2              147       2         1%
1996       141             141             2               77       2         1%
1997       456             456           394              455     394        86%
1998       400             400           279              374     279        70%
1999       850             850           774              823     774        91%
2000       837             835           760              805     758        91%
2001      1125            1125           999             1056     999        89%
2002      1143            1142           998             1067     998        87%
2003      1365            1364          1190             1273    1190        87%
2004      1101            1101           972             1062     972        88%
2005      1521            15

## Step 9. Save the tables

The list columns survive the Parquet round trip as arrays. The hearing table stays the single source, and the two pair tables stay derived.

In [6]:
hearings.to_parquet(INTERIM_FOLDER / "hearings.parquet", index=False)
hearing_committees.to_parquet(INTERIM_FOLDER / "hearing_committees.parquet", index=False)
hearing_members.to_parquet(INTERIM_FOLDER / "hearing_members.parquet", index=False)
coverage.to_csv(INTERIM_FOLDER / "coverage_by_year.csv")

for name in ("hearings", "hearing_committees", "hearing_members"):
    path = INTERIM_FOLDER / f"{name}.parquet"
    print(f"{path.name:28s} {path.stat().st_size / 1e6:>7.1f} MB")

hearings.parquet                12.4 MB
hearing_committees.parquet       0.4 MB
hearing_members.parquet          3.6 MB


## Step 10. Count the hearings of the House Science Committee under its three names

The committee was renamed twice. Selecting it by its current name drops two thirds of its history, so the three names are listed with their year ranges. GovInfo's committee strings carry stray spaces, so names are normalized before matching and the rescued rows are counted.

In [7]:
LINEAGE = [
    "Committee on Science",
    "Committee on Science and Technology",
    "Committee on Science, Space, and Technology",
]

normalized = hearing_committees["name_standard"].str.replace(r"\s+", " ", regex=True).str.strip()
exact = hearing_committees["name_standard"].isin(LINEAGE)
loose = normalized.isin(LINEAGE)
print(f"whitespace normalization rescued {int((loose & ~exact).sum())} committee rows")

lineage = hearing_committees.loc[loose, ["packageId", "name_standard", "authorityId"]].copy()
lineage["committee"] = normalized[loose]
lineage = lineage.merge(hearings[["packageId", "year", "has_transcript", "n_members"]], on="packageId")

print()
print(lineage.groupby("committee").agg(
    hearings=("packageId", "nunique"),
    first_year=("year", "min"),
    last_year=("year", "max"),
    with_transcript=("has_transcript", "sum"),
).to_string())
print(f"\nlineage total: {lineage['packageId'].nunique():,} hearings")
print()
print(lineage.groupby("year")["packageId"].nunique().to_string())

whitespace normalization rescued 0 committee rows

                                             hearings  first_year  last_year  with_transcript
committee                                                                                    
Committee on Science                              147        1996       2006              146
Committee on Science and Technology               231        1998       2010              224
Committee on Science, Space, and Technology       586        2011       2025              589

lineage total: 964 hearings

year
1996     1
1997     3
1998     4
1999    17
2000     2
2001     1
2003    37
2004    31
2005    34
2006    31
2007    62
2008    47
2009    66
2010    42
2011    56
2012    54
2013    56
2014    40
2015    55
2016    41
2017    44
2018    31
2019    61
2020    22
2021    41
2022    32
2023    29
2024    17
2025     7


## Step 11. Check the tables

Five checks. The first four stop the notebook if they fail; the fifth reports coverage.

- No MODS record failed to parse.
- One row per package, and none lost between parsing and the table.
- The member table reconciles with `n_members` on every row.
- `has_transcript` is never true for a package with no file on disk.
- Committee, member, and transcript coverage are reported with the worst years.

In [8]:
assert not failures, f"{len(failures)} MODS records failed to parse; first: {failures[0]}"
print(f"1) all {len(hearings):,} MODS records parsed")

dupes = int(hearings["packageId"].duplicated().sum())
assert dupes == 0, f"{dupes} duplicate packageId in the hearing table"
assert len(hearings) == len(mods_ok), f"{len(mods_ok):,} MODS files but {len(hearings):,} rows: records lost after parsing"
print(f"2) {len(hearings):,} rows, one per package, none lost")

from_pairs = hearing_members.groupby("packageId").size()
from_row = hearings.set_index("packageId")["n_members"]
diff = from_row.subtract(from_pairs, fill_value=0)
assert (diff == 0).all(), f"member counts disagree for {int((diff != 0).sum())} hearings"
print(f"3) member table reconciles with n_members on every row")

bad_flag = int((hearings["has_transcript"] & ~hearings["html_on_disk"]).sum())
assert bad_flag == 0, f"{bad_flag} hearings flagged as having text with no file on disk"
print("4) has_transcript implies the file is on disk")

no_committee = int((hearings["n_committees"] == 0).sum())
no_members = int((hearings["n_members"] == 0).sum())
no_text = int((~hearings["has_transcript"]).sum())
usable = int(((hearings["n_committees"] > 0) & (hearings["n_members"] > 0) & hearings["has_transcript"]).sum())
print(f"5) coverage over {len(hearings):,} hearings")
print(f"     no committee : {no_committee:>6,} ({no_committee / len(hearings):.1%})")
print(f"     no members   : {no_members:>6,} ({no_members / len(hearings):.1%})")
print(f"     no transcript: {no_text:>6,} ({no_text / len(hearings):.1%})")
print(f"     usable       : {usable:>6,} ({usable / len(hearings):.1%})")
print()
worst = coverage.nsmallest(5, "usable")[["hearings", "usable", "usable_pct"]]
print("   years with the least usable coverage:")
print(worst.to_string())

1) all 36,378 MODS records parsed
2) 36,378 rows, one per package, none lost
3) member table reconciles with n_members on every row
4) has_transcript implies the file is on disk
5) coverage over 36,378 hearings
     no committee :     12 (0.0%)
     no members   :  4,448 (12.2%)
     no transcript:  1,253 (3.4%)
     usable       : 31,916 (87.7%)

   years with the least usable coverage:
      hearings  usable usable_pct
year                             
1995       253       2         1%
1996       141       2         1%
1998       400     279        70%
1997       456     394        86%
2020       689     638        93%
